# GRPO E2E 스캘핑 훈련 (Google Colab GPU)

이 노트북은 `train_e2e.py`를 사용하여 강화학습 기반 스캘핑 봇을 Colab GPU에서 훈련합니다.

**훈련 설정:**
- `--seq_len 3000`, `--features 28`
- `--transaction_cost 0.00015` (Curriculum으로 자동 증가)
- `--entropy_coef 0.15`, `--no_trade_penalty 50.0`
- `--min_holding 2`, `--max_holding 300`

**사전 준비:**
1. Colab 왼쪽 자물쇠 아이콘 → Secret 추가
   - `GITHUB_TOKEN`: GitHub Personal Access Token (private repo 접근용)
2. Google Drive에 DB 파일 업로드:
   - `MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb`

## 1. GPU 확인

In [ ]:
import torch
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {mem:.1f} GB")
else:
    print("⚠️ GPU를 사용할 수 없습니다. 런타임 → 런타임 유형 변경 → GPU 선택 후 재시작하세요.")

## 2. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 필수 패키지 설치

In [ ]:
!pip install -q duckdb pandas pyarrow python-dotenv stable-baselines3 tensorboard

## 4. 프로젝트 코드 준비 (GitHub Clone)
Colab Secret에 `GITHUB_TOKEN`이 설정되어 있으면 private repo를 자동으로 클론합니다.

In [ ]:
import os
import sys
from google.colab import userdata

repo_path = '/content/stock-bot2'

try:
    token = userdata.get('GITHUB_TOKEN')
    use_token = True
    print("🔑 GitHub Token 확인됨")
except:
    use_token = False
    print("⚠️ GITHUB_TOKEN 없음. public 클론 시도")

if not os.path.exists(repo_path):
    print("📥 저장소 클론 중...")
    if use_token:
        !git clone https://{token}@github.com/gblue1223/stock-bot2.git {repo_path}
    else:
        !git clone https://github.com/gblue1223/stock-bot2.git {repo_path}
    print("✅ 클론 완료!")
else:
    print("📁 저장소 이미 존재 → 업데이트 중...")
    %cd {repo_path}
    !git fetch origin && git pull
    print("✅ 업데이트 완료!")

%cd {repo_path}
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f"📂 작업 디렉토리: {os.getcwd()}")

## 5. 데이터베이스 준비 (Drive → Local 복사)
Google Drive에서 직접 읽으면 IO 속도가 매우 느립니다.
훈련 전에 Colab 로컬(`/content`)로 복사합니다.

In [ ]:
import shutil

DRIVE_DB_PATH = '/content/drive/MyDrive/ColabData/datasets/stockbot/20260117/datasets_raw_09_11.duckdb'
LOCAL_DB_PATH = '/content/datasets_raw_09_11.duckdb'

if not os.path.exists(LOCAL_DB_PATH):
    if not os.path.exists(DRIVE_DB_PATH):
        raise FileNotFoundError(f"❌ Drive DB 없음: {DRIVE_DB_PATH}\nDrive에 DB를 먼저 업로드 해주세요.")
    print("💾 DB 파일 복사 중... (수 분 소요될 수 있습니다)")
    shutil.copy(DRIVE_DB_PATH, LOCAL_DB_PATH)
    size_gb = os.path.getsize(LOCAL_DB_PATH) / 1e9
    print(f"✅ DB 복사 완료! ({size_gb:.2f} GB)")
else:
    size_gb = os.path.getsize(LOCAL_DB_PATH) / 1e9
    print(f"✅ 로컬 DB 이미 존재 ({size_gb:.2f} GB)")

## 6. 훈련 출력 디렉토리 설정
체크포인트와 로그를 Google Drive에 저장하여 Colab 세션이 종료되어도 유실되지 않도록 합니다.

In [ ]:
# ⚙️ 필요에 따라 경로를 수정하세요
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/ColabData/stockbot/models/grpo_scalping_v10'

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_OUTPUT_DIR}/checkpoints', exist_ok=True)
print(f"✅ 출력 디렉토리: {DRIVE_OUTPUT_DIR}")

## 7. 훈련 시작
처음 훈련하는 경우 `--load_policy` 인자를 제거하거나 비워두세요.
재개하는 경우 `checkpoint_iter<N>.pt` 경로를 지정하세요.

In [ ]:
# ── 설정 ─────────────────────────────────────────
DB_PATH        = LOCAL_DB_PATH
OUTPUT_DIR     = DRIVE_OUTPUT_DIR
TOTAL_TIMESTEPS = 200000
SEQ_LEN        = 3000
FEATURES       = 28
MIN_HOLDING    = 2
MAX_HOLDING    = 300
early_exit_penalty = 1.2
num_workers    = 10
ENTROPY_COEF   = 0.15
TRANSACTION_COST = 0.00215  # Curriculum이 자동으로 증가 (0 → 0.00215)
NO_TRADE_PENALTY = 30.0
LR             = 0.00001
LOAD_POLICY    = f'{DRIVE_OUTPUT_DIR}/checkpoints/checkpoint_iter770.pt'  # 재개 시: '/content/drive/.../checkpoint_iter500.pt'
# ─────────────────────────────────────────────────

load_policy_arg = f'--load_policy "{LOAD_POLICY}"' if LOAD_POLICY else ''

cmd = f'''
python ai_trader/grpo/train_e2e.py \n    --db "{DB_PATH}" \n    {load_policy_arg} \n    --vram_preset large \n    --seq_len {SEQ_LEN} \n    --features {FEATURES} \n    --total_timesteps {TOTAL_TIMESTEPS} \n    --output_dir "{OUTPUT_DIR}" \n    --min_holding {MIN_HOLDING} \n    --max_holding {MAX_HOLDING} \n    --early_exit_penalty {early_exit_penalty} \n    --num_workers {num_workers} \n    --entropy_coef {ENTROPY_COEF} \n    --transaction_cost {TRANSACTION_COST} \n    --no_trade_penalty {NO_TRADE_PENALTY} \n    --lr {LR}
'''

print("🚀 훈련 시작...")
print(f"명령어:
{cmd}")
!{cmd}

## 8. 훈련 모니터링 (TensorBoard)

In [ ]:
%load_ext tensorboard
TENSORBOARD_LOG_DIR = f'{DRIVE_OUTPUT_DIR}/tensorboard_logs'
%tensorboard --logdir {TENSORBOARD_LOG_DIR}

## 9. 로그 분석 (선택)
훈련 중 또는 후에 실행하여 지표를 요약합니다.

In [ ]:
import re
import glob

def parse_log(filepath):
    pattern = re.compile(
        r"Iteration (\d+)/.*?Mean Reward: ([\-\d.]+) \| Win Rate: ([\d.]+)% \| "
        r"Trades: (\d+) \| Sharpe: ([\-\d.]+) \| AvgHold: ([\d.]+)s"
    )
    metrics = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            m = pattern.search(line)
            if m:
                metrics.append({
                    'iter': int(m.group(1)), 'reward': float(m.group(2)),
                    'win_rate': float(m.group(3)), 'trades': int(m.group(4)),
                    'sharpe': float(m.group(5)), 'avg_hold': float(m.group(6))
                })
    return metrics

log_files = sorted(glob.glob('logs/train_scalping_*.log'), reverse=True)
if log_files:
    latest = log_files[0]
    print(f"📋 최신 로그: {latest}")
    data = parse_log(latest)
    if data:
        print(f"\n총 {len(data)} iterations 파싱 완료 ({data[0]['iter']} → {data[-1]['iter']})")
        print(f"\n최근 100 iteration 평균:")
        recent = data[-100:]
        print(f"  Mean Reward : {sum(x['reward'] for x in recent)/len(recent):.4f}")
        print(f"  Win Rate    : {sum(x['win_rate'] for x in recent)/len(recent):.2f}%")
        print(f"  Avg Trades  : {sum(x['trades'] for x in recent)/len(recent):.1f}")
        print(f"  Avg Sharpe  : {sum(x['sharpe'] for x in recent)/len(recent):.3f}")
        print(f"  Avg Hold    : {sum(x['avg_hold'] for x in recent)/len(recent):.2f}s")
    else:
        print("❌ 아직 iteration 로그가 없습니다. 훈련을 먼저 시작하세요.")
else:
    print("❌ 로그 파일이 없습니다.")